# Basic-Lite TRACE — Pipeline de Entrenamiento Multi-Hardware (Hito 3.6)
**Autor e Inventor:** Miguel Angel Aranguren Ramirez (Miguel19x)  
**Régimen de Licencia:** Source-Available ([LICENSE.txt](LICENSE.txt)) para investigación, docencia y evaluación técnica. Exclusión de explotación comercial directa y destilación no autorizada.  
**Hardware Objetivo:** Compatible con cualquier GPU PyTorch (Colab T4/L4/A100, Kaggle 1×/2× T4 con DDP, ROCm, MPS, XPU y CPU).

In [ ]:
# [Celda 0] Detección de Entorno, Configuración de Rutas y Persistencia (Colab vs Kaggle)
import os
import subprocess
import sys

# 1. Detección de plataforma
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = os.path.exists("/kaggle/working")

print(f"Entorno detectado: {'Google Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Local / Linux')}")

# 2. Clonar rama exp/fase3-language-mla si no estamos dentro del repositorio
REPO_NAME = "Basic-Lite-TRACE"
if not os.path.exists("trace") and not os.path.exists("csrc"):
    if not os.path.exists(REPO_NAME):
        print("Clonando repositorio Basic-Lite TRACE (rama exp/fase3-language-mla)...")
        subprocess.check_call(["git", "clone", "-b", "exp/fase3-language-mla", "https://github.com/Miguel19x/Basic-Lite-TRACE.git"])
    os.chdir(REPO_NAME)
print(f"Directorio de trabajo actual: {os.getcwd()}")

# 3. Instalación de dependencias periféricas
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "ruff"])

# 4. Montaje opcional de Google Drive en Colab para persistencia de checkpoints y resultados
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado en /content/drive")
        DRIVE_BACKUP_DIR = "/content/drive/MyDrive/basic_lite_trace_checkpoints"
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print(f"Directorio de respaldo en Drive: {DRIVE_BACKUP_DIR}")
    except Exception as e:
        print(f"Aviso: No se montó Drive ({e}). Los checkpoints se guardarán en almacenamiento local efímero.")
elif IS_KAGGLE:
    print("✓ Ejecutando en Kaggle. Los artefactos de salida se guardarán en /kaggle/working/")

In [ ]:
# [Celda 1] Verificación de Hardware y Soporte de Precisión
!nvidia-smi
import torch

print(f"PyTorch: {torch.__version__} | CUDA Disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    bf16_ok = torch.cuda.is_bf16_supported()
    device_count = torch.cuda.device_count()
    print(f"GPU(s): {device_count}x {device_name} | Soporte Nativo BF16: {bf16_ok}")

In [ ]:
# [Celda 2] Compilación de Bibliotecas y Motor Nativo C Standalone
!make clean && make all

In [ ]:
# [Celda 3] Preparación de Corpus y Entrenamiento del Tokenizador BPE 4K
# Sube --max-docs a 300000 (~80-100M tokens, ~3-4 épocas) para evitar sobreajuste en el piloto de 5000 pasos
!python3 scripts/prepare_lm_corpus.py \
    --dataset roneneldan/TinyStories \
    --max-docs 300000 \
    --train-tokenizer \
    --target-vocab-size 4096 \
    --output-dir data


In [ ]:
# [Celda 4] Lanzamiento del Entrenamiento Multi-Hardware
import torch

device_count = torch.cuda.device_count() if torch.cuda.is_available() else 1

if device_count > 1:
    print(f"Lanzando entrenamiento distribuido DDP sobre {device_count} GPUs (Kaggle 2x T4)...")
    # 2 GPUs * 8 batch_size * 4 grad_accum * 1024 seq_len = 65,536 tokens/paso (327M tokens en 5,000 pasos)
    !torchrun --nproc_per_node={device_count} scripts/train_lm.py \
        --preset 25m \
        --max-seq-len 1024 \
        --batch-size 8 \
        --grad-accum-steps 4 \
        --lr 1e-3 \
        --weight-decay 0.05 \
        --fp16-warmup-ratio 0.15 \
        --max-steps 5000 \
        --weight-scale mean \
        --compile \
        --log-every 20 \
        --eval-every 250 \
        --save-every 500
else:
    print("Lanzando entrenamiento sobre 1 GPU (Colab T4 / L4 / A100)...")
    # 1 GPU * 8 batch_size * 8 grad_accum * 1024 seq_len = 65,536 tokens/paso (327M tokens en 5,000 pasos)
    !python3 scripts/train_lm.py \
        --preset 25m \
        --max-seq-len 1024 \
        --batch-size 8 \
        --grad-accum-steps 8 \
        --lr 1e-3 \
        --weight-decay 0.05 \
        --fp16-warmup-ratio 0.15 \
        --max-steps 5000 \
        --weight-scale mean \
        --compile \
        --log-every 20 \
        --eval-every 250 \
        --save-every 500


In [ ]:
# [Celda 5] Verificación de Paridad Formal y Falsación Activa de Fase 3
!python3 tests/test_parity_fase3.py
!python3 tests/test_mla_language_model.py

In [ ]:
# [Celda 6] Exportación a Paged BitCOS v1.1 (0 pesos FP32 en RAM, O_DIRECT ready)
!python3 export_paged_lm.py \
    --checkpoint checkpoints/best_model.pt \
    --output checkpoints/trace_lm_trained_paged_bitcos.bin

In [ ]:
# [Celda 7] Inferencia Standalone Nativa en C con Decodificación de Ultra-Alta Velocidad
!./trace_lm \
    --model checkpoints/trace_lm_trained_paged_bitcos.bin \
    --tokenizer checkpoints/trace_tokenizer_4k.bin \
    --prompt "Once upon a time" \
    --max_tokens 32 \
    --temp 0.0